In [ ]:
# STEP 1: Load raw data

import pandas as pd

FILE_PATH = "/home/aaryajain/exoplanet_project/PSCompPars_2026.03.05_02.02.54.csv"

df_raw = pd.read_csv(FILE_PATH, comment="#")
print("Loaded successfully")
print(f"Shape: {df_raw.shape[0]} planets x {df_raw.shape[1]} columns")

df_raw.to_csv("exoplanets_raw_backup.csv", index=False)
print("Raw backup saved -> exoplanets_raw_backup.csv")

print(df_raw.head(3))
print(df_raw.describe())

Loaded successfully
Shape: 6128 planets x 54 columns
Raw backup saved -> exoplanets_raw_backup.csv
    pl_name hostname  pl_orbper  pl_orbpererr1  pl_orbpererr2  pl_orbperlim  \
0  11 Com b   11 Com  323.21000           0.06          -0.05           0.0   
1  11 UMi b   11 UMi  516.21997           3.20          -3.20           0.0   
2  14 And b   14 And  186.76000           0.11          -0.12           0.0   

   pl_orbsmax  pl_orbsmaxerr1  pl_orbsmaxerr2  pl_orbsmaxlim  ...  \
0       1.178            0.00            0.00            0.0  ...   
1       1.530            0.07           -0.07            0.0  ...   
2       0.775            0.00            0.00            0.0  ...   

   st_masserr2  st_masslim   st_lum  st_lumerr1  st_lumerr2  st_lumlim  \
0        -0.63         0.0  1.97823     0.18002    -0.15868        0.0   
1        -0.69         0.0  2.42951     0.00801    -0.00816        0.0   
2        -0.29         0.0  1.83992     0.08135    -0.04991        0.0   

   st_age 

In [ ]:
# STEP 2: Select core columns

ID_COLS = ["pl_name", "hostname"]
PLANETARY_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt", "pl_insol"]
ORBITAL_COLS = ["pl_orbper", "pl_orbsmax"]
STELLAR_COLS = ["st_teff", "st_lum", "st_rad", "st_mass", "st_age"]

ALL_KEEP = ID_COLS + PLANETARY_COLS + ORBITAL_COLS + STELLAR_COLS

missing_cols = [c for c in ALL_KEEP if c not in df_raw.columns]
available_cols = [c for c in ALL_KEEP if c in df_raw.columns]

if missing_cols:
    print(f"Columns not found: {missing_cols}")
else:
    print(f"All {len(ALL_KEEP)} requested columns found")

df = df_raw[available_cols].copy()
print(f"Columns kept: {df.shape[1]} (dropped {df_raw.shape[1] - df.shape[1]})")
print(f"Planets kept: {df.shape[0]}")

missing_report = pd.DataFrame({
    "Column": df.columns,
    "Category": (["ID"] * 2 + ["Planetary"] * 6 + ["Orbital"] * 2 + ["Stellar"] * 5),
    "Non-Null": df.notna().sum().values,
    "Missing": df.isna().sum().values,
    "% Missing": (df.isna().mean() * 100).round(1).values,
    "Action": [
        "keep", "keep",
        "median impute",
        "median impute",
        "median impute",
        "MICE impute",
        "MICE impute",
        "MICE impute",
        "median impute",
        "median impute",
        "median impute",
        "median impute",
        "median impute",
        "median impute",
        "MICE impute",
    ],
}).set_index("Column")

print(missing_report.to_string())

df.to_csv("exoplanets_selected.csv", index=False)
print("Selected dataset saved -> exoplanets_selected.csv")
print(f"Final shape: {df.shape[0]} planets x {df.shape[1]} columns")

All 15 requested columns found
Columns kept: 15 (dropped 39)
Planets kept: 6128
              Category  Non-Null  Missing  % Missing         Action
Column                                                             
pl_name             ID      6128        0        0.0           keep
hostname            ID      6128        0        0.0           keep
pl_rade      Planetary      6078       50        0.8  median impute
pl_bmasse    Planetary      6097       31        0.5  median impute
pl_dens      Planetary      5989      139        2.3  median impute
pl_orbeccen  Planetary      5200      928       15.1    MICE impute
pl_eqt       Planetary      4580     1548       25.3    MICE impute
pl_insol     Planetary      4304     1824       29.8    MICE impute
pl_orbper      Orbital      5799      329        5.4  median impute
pl_orbsmax     Orbital      5812      316        5.2  median impute
st_teff        Stellar      5843      285        4.7  median impute
st_lum         Stellar      5825    

In [ ]:
# STEP 3: Impute missing values

import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("exoplanets_selected.csv")
print(f"Loaded: {df.shape[0]} planets x {df.shape[1]} columns")

FEATURE_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt",
                "pl_insol", "pl_orbper", "pl_orbsmax",
                "st_teff", "st_lum", "st_rad", "st_mass", "st_age"]

missing_pct = (df[FEATURE_COLS].isna().mean() * 100).round(2)
print("Missing percentage by feature:")
print(missing_pct.sort_values(ascending=False).to_string())

LOG_COLS = ["pl_orbper", "pl_orbsmax", "pl_bmasse", "pl_dens",
            "pl_insol", "st_rad", "st_mass"]

df_work = df.copy()

for col in LOG_COLS:
    min_val = df_work[col].min()
    shift = abs(min_val) + 1e-6 if min_val <= 0 else 0
    df_work[f"{col}_log"] = np.log1p(df_work[col] + shift)

print("Log-transformed columns:", LOG_COLS)

FEATURE_WORK = []
for col in FEATURE_COLS:
    FEATURE_WORK.append(f"{col}_log" if col in LOG_COLS else col)

print("Working features:", FEATURE_WORK)

MEDIAN_COLS_ORIG = ["pl_rade", "pl_orbper", "pl_orbsmax",
                    "pl_bmasse", "pl_dens", "st_teff",
                    "st_lum", "st_rad", "st_mass"]

MEDIAN_COLS_WORK = [f"{c}_log" if c in LOG_COLS else c for c in MEDIAN_COLS_ORIG]

median_imputer = SimpleImputer(strategy="median")
df_work[MEDIAN_COLS_WORK] = median_imputer.fit_transform(df_work[MEDIAN_COLS_WORK])

print(f"Median imputation done for {len(MEDIAN_COLS_WORK)} columns")
for orig, work in zip(MEDIAN_COLS_ORIG, MEDIAN_COLS_WORK):
    print(f"{orig:15s} -> imputed via {work}")

MICE_COLS_ORIG = ["pl_orbeccen", "pl_eqt", "pl_insol", "st_age"]
MICE_COLS_WORK = [f"{c}_log" if c in LOG_COLS else c for c in MICE_COLS_ORIG]

mice_input = df_work[FEATURE_WORK].copy()

print("Running MICE imputation")
mice_imputer = IterativeImputer(
    estimator=RandomForestRegressor(n_estimators=10, random_state=42),
    max_iter=10,
    random_state=42,
    verbose=1
)
mice_output = mice_imputer.fit_transform(mice_input)
mice_df = pd.DataFrame(mice_output, columns=FEATURE_WORK)

for col in MICE_COLS_WORK:
    df_work[col] = mice_df[col].values

print(f"MICE imputation done for: {MICE_COLS_ORIG}")

for col in LOG_COLS:
    log_col = f"{col}_log"
    df_work[col] = np.expm1(df_work[log_col])
    df_work.drop(columns=[log_col], inplace=True)

print("Log transforms reversed")

remaining_missing = df_work[FEATURE_COLS].isna().sum()
print(remaining_missing)

if remaining_missing.sum() == 0:
    print("All missing values handled")
else:
    print(f"{remaining_missing.sum()} missing values remain")

df_clean = df_work[["pl_name", "hostname"] + FEATURE_COLS].copy()
df_clean.to_csv("exoplanets_step3_clean.csv", index=False)

print("Clean dataset saved -> exoplanets_step3_clean.csv")
print(f"Final shape: {df_clean.shape[0]} planets x {df_clean.shape[1]} columns")
print(df_clean[FEATURE_COLS].describe().round(3))

Loaded: 6128 planets x 15 columns
Missing percentage by feature:
pl_insol       29.77
pl_eqt         25.26
st_age         21.20
pl_orbeccen    15.14
pl_orbper       5.37
pl_orbsmax      5.16
st_rad          5.04
st_lum          4.94
st_teff         4.65
pl_dens         2.27
pl_rade         0.82
pl_bmasse       0.51
st_mass         0.13
Log-transformed columns: ['pl_orbper', 'pl_orbsmax', 'pl_bmasse', 'pl_dens', 'pl_insol', 'st_rad', 'st_mass']
Working features: ['pl_rade', 'pl_bmasse_log', 'pl_dens_log', 'pl_orbeccen', 'pl_eqt', 'pl_insol_log', 'pl_orbper_log', 'pl_orbsmax_log', 'st_teff', 'st_lum', 'st_rad_log', 'st_mass_log', 'st_age']
Median imputation done for 9 columns
pl_rade         -> imputed via pl_rade
pl_orbper       -> imputed via pl_orbper_log
pl_orbsmax      -> imputed via pl_orbsmax_log
pl_bmasse       -> imputed via pl_bmasse_log
pl_dens         -> imputed via pl_dens_log
st_teff         -> imputed via st_teff
st_lum          -> imputed via st_lum
st_rad          -> imp

/home/aaryajain/exoplanet_project/.venv/lib/python3.12/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [ ]:
# STEP 4: Apply outlier filters

import pandas as pd
import numpy as np

df = pd.read_csv("exoplanets_step3_clean.csv")
print(f"Loaded: {df.shape[0]} planets x {df.shape[1]} columns")

FEATURE_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt",
                "pl_insol", "pl_orbper", "pl_orbsmax",
                "st_teff", "st_lum", "st_rad", "st_mass", "st_age"]

df_clean = df.copy()
removal_log = []

domain_rules = {
    "pl_bmasse": (0.1, 4131.0, "Brown dwarf boundary"),
    "pl_rade": (0.3, 25.0, "Radius bounds"),
    "pl_dens": (0.001, 100.0, "Density bounds"),
    "pl_orbper": (0.1, 100000, "Orbital period bounds"),
    "pl_orbsmax": (0.001, 100.0, "Semi-major axis bounds"),
    "pl_eqt": (50, 4000.0, "Equilibrium temperature bounds"),
    "pl_insol": (0.0, 30000.0, "Insolation bounds"),
    "pl_orbeccen": (0.0, 0.95, "Eccentricity bounds"),
    "st_teff": (2000, 40000.0, "Stellar temperature bounds"),
    "st_rad": (0.01, 50.0, "Stellar radius bounds"),
    "st_mass": (0.05, 8.0, "Stellar mass bounds"),
    "st_age": (0.0, 14.0, "Stellar age bounds"),
    "st_lum": (-6.0, 3.8, "Stellar luminosity bounds"),
}

print("DOMAIN-BASED REMOVAL")
print(f"{'Feature':<15} {'Rule':<25} {'Removed':>8}  Reason")
print("-" * 90)

for col, (lo, hi, reason) in domain_rules.items():
    if col not in df_clean.columns:
        continue
    mask_out = (df_clean[col] < lo) | (df_clean[col] > hi)
    n_removed = mask_out.sum()
    removal_log.append({
        "step": "domain",
        "feature": col,
        "rule": f"[{lo}, {hi}]",
        "removed": n_removed,
        "reason": reason
    })
    df_clean = df_clean[~mask_out].copy()
    print(f"{col:<15} [{lo}, {hi}]     {n_removed:>6} rows   {reason}")

print(f"After domain cuts: {df_clean.shape[0]} planets remain")

LOG_COLS = ["pl_orbper", "pl_orbsmax", "pl_bmasse", "pl_dens",
            "pl_insol", "st_rad", "st_mass"]

Z_THRESHOLD = 3.5

print(f"Z-SCORE OUTLIER REMOVAL (threshold = +/-{Z_THRESHOLD})")
print(f"{'Feature':<15} {'Removed':>8}  {'Remaining':>10}")
print("-" * 40)

before_z = df_clean.shape[0]

for col in FEATURE_COLS:
    if col not in df_clean.columns:
        continue
    vals = np.log1p(df_clean[col]) if col in LOG_COLS else df_clean[col]
    z_scores = (vals - vals.mean()) / vals.std()
    mask_out = z_scores.abs() > Z_THRESHOLD
    n_removed = mask_out.sum()

    if n_removed > 0:
        removal_log.append({
            "step": "zscore",
            "feature": col,
            "rule": f"|z| > {Z_THRESHOLD}",
            "removed": n_removed,
            "reason": "Statistical outlier on log-transformed data"
        })
        df_clean = df_clean[~mask_out].copy()
        print(f"{col:<15} {n_removed:>8}  {df_clean.shape[0]:>10}")

print(f"After Z-score cuts: {df_clean.shape[0]} planets remain")

total_removed = df.shape[0] - df_clean.shape[0]
pct_removed = total_removed / df.shape[0] * 100

print("FULL REMOVAL SUMMARY")
print(f"Original rows: {df.shape[0]}")
print(f"Rows removed: {total_removed} ({pct_removed:.1f}%)")
print(f"Rows remaining: {df_clean.shape[0]}")

removal_df = pd.DataFrame(removal_log)
print(removal_df.to_string(index=False))

df_clean.reset_index(drop=True, inplace=True)
df_clean.to_csv("exoplanets_step4_clean.csv", index=False)

print("Clean dataset saved -> exoplanets_step4_clean.csv")
print(f"Final shape: {df_clean.shape[0]} planets x {df_clean.shape[1]} columns")
print(df_clean[FEATURE_COLS].describe().round(3))

Loaded: 6128 planets x 15 columns
DOMAIN-BASED REMOVAL
Feature         Rule                       Removed  Reason
------------------------------------------------------------------------------------------
pl_bmasse       [0.1, 4131.0]        163 rows   Brown dwarf boundary
pl_rade         [0.3, 25.0]          5 rows   Radius bounds
pl_dens         [0.001, 100.0]         14 rows   Density bounds
pl_orbper       [0.1, 100000]          6 rows   Orbital period bounds
pl_orbsmax      [0.001, 100.0]         20 rows   Semi-major axis bounds
pl_eqt          [50, 4000.0]          3 rows   Equilibrium temperature bounds
pl_insol        [0.0, 30000.0]          1 rows   Insolation bounds
pl_orbeccen     [0.0, 0.95]          0 rows   Eccentricity bounds
st_teff         [2000, 40000.0]          4 rows   Stellar temperature bounds
st_rad          [0.01, 50.0]          2 rows   Stellar radius bounds
st_mass         [0.05, 8.0]          9 rows   Stellar mass bounds
st_age          [0.0, 14.0]          

In [ ]:
# STEP 5: Apply log transforms

import pandas as pd
import numpy as np
import json

df = pd.read_csv("exoplanets_step4_clean.csv")
print(f"Loaded: {df.shape}")

FEATURE_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt",
                "pl_insol", "pl_orbper", "pl_orbsmax",
                "st_teff", "st_lum", "st_rad", "st_mass", "st_age"]

df_out = df.copy()
SKEW_THRESHOLD = 1.0
LOG_TRANSFORMED = {}

print(f"{'Feature':<15} {'Skew Before':>12} {'Skew After':>12} {'Drop':>8}  Decision")
print("-" * 65)

for col in FEATURE_COLS:
    if col == "st_lum":
        LOG_TRANSFORMED[col] = False
        print(f"{col:<15} {'(already log10)':>12}  SKIP")
        continue

    skew_before = df_out[col].skew()
    skew_after = np.log1p(df_out[col]).skew()
    improvement = abs(skew_before) - abs(skew_after)

    if improvement >= SKEW_THRESHOLD:
        df_out[f"{col}_log"] = np.log1p(df_out[col])
        df_out.drop(columns=[col], inplace=True)
        df_out.rename(columns={f"{col}_log": col}, inplace=True)
        LOG_TRANSFORMED[col] = True
        decision = f"LOG1P  (dskew={improvement:+.2f})"
    else:
        LOG_TRANSFORMED[col] = False
        decision = f"KEEP   (dskew={improvement:+.2f} < threshold)"

    print(f"{col:<15} {skew_before:>12.3f} {skew_after:>12.3f} {improvement:>8.3f}  {decision}")

with open("transform_metadata.json", "w") as f:
    json.dump(LOG_TRANSFORMED, f, indent=2)

print("Transform metadata saved -> transform_metadata.json")
print(f"Log-transformed: {[k for k, v in LOG_TRANSFORMED.items() if v]}")
print(f"Kept as-is: {[k for k, v in LOG_TRANSFORMED.items() if not v]}")

df_out.to_csv("exoplanets_step5_transformed.csv", index=False)
print("Saved -> exoplanets_step5_transformed.csv")

Loaded: (5456, 15)
Feature          Skew Before   Skew After     Drop  Decision
-----------------------------------------------------------------
pl_rade                1.182        0.682    0.500  KEEP   (dskew=+0.50 < threshold)
pl_bmasse              4.659        1.002    3.657  LOG1P  (dskew=+3.66)
pl_dens                2.376        0.008    2.368  LOG1P  (dskew=+2.37)
pl_orbeccen            2.150        1.930    0.220  KEEP   (dskew=+0.22 < threshold)
pl_eqt                 0.840       -0.739    0.101  KEEP   (dskew=+0.10 < threshold)
pl_insol               5.613       -0.092    5.521  LOG1P  (dskew=+5.52)
pl_orbper              7.429        1.351    6.079  LOG1P  (dskew=+6.08)
pl_orbsmax             3.527        2.702    0.825  KEEP   (dskew=+0.83 < threshold)
st_teff               -0.947       -1.341   -0.393  KEEP   (dskew=-0.39 < threshold)
st_lum          (already log10)  SKIP
st_rad                 3.701        1.483    2.218  LOG1P  (dskew=+2.22)
st_mass               -0.2

In [ ]:
# STEP 6: Perform VIF reduction

import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

df = pd.read_csv("exoplanets_step5_transformed.csv")
df_step4 = pd.read_csv("exoplanets_step4_clean.csv")
df["st_teff"] = df_step4["st_teff"].values
df["st_mass"] = df_step4["st_mass"].values

print(f"Loaded: {df.shape[0]} planets x {df.shape[1]} columns")
print("st_teff and st_mass reverted to original scale")

FEATURE_COLS = ["pl_rade", "pl_bmasse", "pl_dens", "pl_orbeccen", "pl_eqt",
                "pl_insol", "pl_orbper", "pl_orbsmax",
                "st_teff", "st_lum", "st_rad", "st_mass", "st_age"]

corr = df[FEATURE_COLS].corr()

print("CORRELATION PAIRS REPORT")
print(f"{'Feature A':<15} {'Feature B':<15} {'r':>8}  Level")
print("-" * 55)

pairs = []
for i in range(len(FEATURE_COLS)):
    for j in range(i + 1, len(FEATURE_COLS)):
        r = corr.iloc[i, j]
        a, b = FEATURE_COLS[i], FEATURE_COLS[j]
        if abs(r) >= 0.5:
            level = "HIGH" if abs(r) >= 0.7 else "MODERATE"
            pairs.append({"a": a, "b": b, "r": r, "level": level})
            print(f"{a:<15} {b:<15} {r:>8.3f}  {level}")

print(f"Total flagged pairs: {len(pairs)}")

def compute_vif(dataframe, features):
    X = dataframe[features].dropna()
    X_const = add_constant(X)
    vif_data = pd.DataFrame()
    vif_data["Feature"] = features
    vif_data["VIF"] = [
        variance_inflation_factor(X_const.values, i + 1)
        for i in range(len(features))
    ]
    vif_data["Status"] = vif_data["VIF"].apply(
        lambda v: "DROP" if v > 10 else "MONITOR" if v > 5 else "OK"
    )
    return vif_data.sort_values("VIF", ascending=False)

print("VIF - FULL FEATURE SET (BEFORE REMOVAL)")
vif_before = compute_vif(df, FEATURE_COLS)
print(vif_before.to_string(index=False))

DROP_FEATURES = ["pl_bmasse", "pl_insol", "pl_orbper", "st_lum", "st_rad"]
KEEP_FEATURES = [f for f in FEATURE_COLS if f not in DROP_FEATURES]

print("DROP DECISION SUMMARY")
print("DROP (reason)")
print("-" * 70)
drop_reasons = {
    "pl_bmasse": "r=0.912 with pl_rade",
    "pl_insol": "r=0.969 with pl_eqt",
    "pl_orbper": "r=0.901 with pl_orbsmax",
    "st_lum": "highly correlated with st_teff and st_rad",
    "st_rad": "highly correlated with st_lum and st_mass",
}
for col, reason in drop_reasons.items():
    print(f"{col:<15} {reason}")

print("KEEP (reason)")
print("-" * 70)
keep_reasons = {
    "pl_rade": "Direct planet size",
    "pl_dens": "Independent physical property",
    "pl_orbeccen": "Orbital stability driver",
    "pl_eqt": "Habitability temperature proxy",
    "pl_orbsmax": "HZ placement parameter",
    "st_teff": "Stellar radiation proxy",
    "st_mass": "Gravitational measure",
    "st_age": "System age proxy",
}
for col, reason in keep_reasons.items():
    print(f"{col:<15} {reason}")

print("VIF - REDUCED FEATURE SET (AFTER REMOVAL)")
vif_after = compute_vif(df, KEEP_FEATURES)
print(vif_after.to_string(index=False))

df_final = df[["pl_name", "hostname"] + KEEP_FEATURES].copy()
df_final.to_csv("exoplanets_step6_final_features.csv", index=False)

print("Saved -> exoplanets_step6_final_features.csv")
print(f"Features in: 13")
print(f"Features out: {len(DROP_FEATURES)} dropped -> {', '.join(DROP_FEATURES)}")
print(f"Features kept: {len(KEEP_FEATURES)} -> {', '.join(KEEP_FEATURES)}")
print(f"Planets: {df_final.shape[0]}")

print("PIPELINE STATUS")
print("Raw (6128x54)")
print("Step 2: select (6128x15)")
print("Step 3: impute (6128x15, zero missing)")
print("Step 4: outliers (5456x15)")
print("Step 5: transforms (5456x15)")
print("Step 6: collinearity (5456x10)")

Loaded: 5456 planets x 15 columns
st_teff and st_mass reverted to original scale
CORRELATION PAIRS REPORT
Feature A       Feature B              r  Level
-------------------------------------------------------
pl_rade         pl_bmasse          0.887  HIGH
pl_rade         pl_dens           -0.620  MODERATE
pl_eqt          pl_insol           0.942  HIGH
pl_eqt          pl_orbper         -0.663  MODERATE
pl_insol        pl_orbper         -0.694  MODERATE
pl_orbper       pl_orbsmax         0.592  MODERATE
st_teff         st_lum             0.898  HIGH
st_teff         st_rad             0.642  MODERATE
st_teff         st_mass            0.786  HIGH
st_lum          st_rad             0.852  HIGH
st_lum          st_mass            0.837  HIGH
st_rad          st_mass            0.790  HIGH
Total flagged pairs: 12
VIF - FULL FEATURE SET (BEFORE REMOVAL)
    Feature       VIF  Status
     st_lum 17.580673    DROP
   pl_insol 13.822017    DROP
     pl_eqt 11.207410    DROP
    st_teff  8.755277 

In [ ]:
# STEP 7: Compute LSS target

import pandas as pd
import numpy as np

df_ml = pd.read_csv("exoplanets_step6_final_features.csv")
df_raw = pd.read_csv("exoplanets_step4_clean.csv")

df_raw = df_raw[df_raw["pl_name"].isin(df_ml["pl_name"])].copy()
df_raw = df_raw.sort_values("pl_name").reset_index(drop=True)
df_ml = df_ml.sort_values("pl_name").reset_index(drop=True)

print(f"ML features: {df_ml.shape}")
print(f"Raw values: {df_raw.shape}")
print("Original unit ranges used for LSS")
for col in ["pl_eqt", "pl_rade", "pl_dens", "pl_orbeccen", "pl_orbsmax", "st_teff", "st_mass", "st_age"]:
    print(f"{col:<15} median={df_raw[col].median():.2f} min={df_raw[col].min():.2f} max={df_raw[col].max():.2f}")

L = 10 ** df_raw["st_lum"]
hz_inner = 0.95 * np.sqrt(L)
hz_outer = 1.67 * np.sqrt(L)
hz_center = (hz_inner + hz_outer) / 2
hz_width = (hz_outer - hz_inner) / 2
hz_dist = np.abs(df_raw["pl_orbsmax"] - hz_center)
hz_score = np.exp(-0.5 * (hz_dist / (hz_width * 1.5)) ** 2)

print(f"hz_score mean={hz_score.mean():.3f} std={hz_score.std():.3f} >0.5: {(hz_score > 0.5).sum()}")

eqt = df_raw["pl_eqt"]
temp_optimal = np.exp(-0.5 * ((eqt - 255.0) / 80.0) ** 2)
temp_partial = np.exp(-0.5 * ((eqt - 255.0) / 400.0) ** 2) * 0.5
temp_score = np.maximum(temp_optimal, temp_partial).clip(0, 1)

print(f"temp_score mean={temp_score.mean():.3f} std={temp_score.std():.3f} >0.5: {(temp_score > 0.5).sum()}")

rade = df_raw["pl_rade"]
dens = df_raw["pl_dens"]
radius_score = np.exp(-0.5 * ((rade - 1.3) / 0.7) ** 2)
density_score = np.clip(dens / 5.51, 0, 1)
size_gate = np.where(rade > 4.0, np.exp(-0.5 * ((rade - 4.0) / 1.5) ** 2), 1.0)
retention_score = (0.5 * radius_score + 0.5 * density_score * size_gate).clip(0, 1)

print(f"retention mean={retention_score.mean():.3f} std={retention_score.std():.3f} >0.5: {(retention_score > 0.5).sum()}")

orbit_score = np.exp(-2.0 * df_raw["pl_orbeccen"]).clip(0, 1)

print(f"orbit_score mean={orbit_score.mean():.3f} std={orbit_score.std():.3f} >0.5: {(orbit_score > 0.5).sum()}")

age = df_raw["st_age"]
teff = df_raw["st_teff"]
mass = df_raw["st_mass"]

age_score = np.where(age < 2.0, age / 2.0,
            np.where(age <= 8.0, 1.0,
            np.clip(1.0 - (age - 8.0) / 6.0, 0, 1)))
teff_score = np.exp(-0.5 * ((teff - 5500.0) / 1500.0) ** 2)
mass_score = np.clip(1.0 - np.maximum(mass - 1.5, 0) / 2.0, 0, 1)
stellar_score = (0.4 * age_score + 0.4 * teff_score + 0.2 * mass_score).clip(0, 1)

print(f"stellar_score mean={stellar_score.mean():.3f} std={stellar_score.std():.3f} >0.5: {(stellar_score > 0.5).sum()}")

W_HZ = 0.35
W_TEMP = 0.25
W_RETENTION = 0.20
W_ORBIT = 0.10
W_STELLAR = 0.10

LSS = (W_HZ * hz_score +
       W_TEMP * temp_score +
       W_RETENTION * retention_score +
       W_ORBIT * orbit_score +
       W_STELLAR * stellar_score).clip(0, 1)

print("LSS distribution")
print(LSS.describe().round(4))
print(f"Spread std: {LSS.std():.4f}")
print(f"Skewness: {LSS.skew():.3f}")
print(f"IQR: {LSS.quantile(0.75) - LSS.quantile(0.25):.4f}")

e = {
    "hz": float(np.exp(-0.5 * ((1.0 - (0.95 + 1.67) / 2) / ((1.67 - 0.95) / 2 * 1.5)) ** 2)),
    "temp": float(np.exp(-0.5 * ((255 - 255) / 80) ** 2)),
    "ret": float(min(0.5 * np.exp(-0.5 * ((1.0 - 1.3) / 0.7) ** 2) + 0.5 * min(5.51 / 5.51, 1) * 1.0, 1)),
    "orb": float(np.exp(-2.0 * 0.017)),
    "st": float(0.4 * 1.0 + 0.4 * np.exp(-0.5 * ((5778 - 5500) / 1500) ** 2) + 0.2 * 1.0),
}
earth_lss = (W_HZ * e["hz"] + W_TEMP * e["temp"] + W_RETENTION * e["ret"] + W_ORBIT * e["orb"] + W_STELLAR * e["st"])

pct = (LSS < earth_lss).mean() * 100
print("Earth benchmark")
for k, v in e.items():
    print(f"{k:<6}: {v:.3f}")
print(f"Earth LSS: {earth_lss:.4f}")
print(f"Percentile: {pct:.1f}th")

results = df_raw[["pl_name", "pl_eqt", "pl_rade", "pl_dens", "pl_orbsmax", "pl_orbeccen"]].copy()
results["hz_score"] = hz_score.values
results["temp_score"] = temp_score.values
results["retention_score"] = retention_score.values
results["orbit_score"] = orbit_score.values
results["stellar_score"] = stellar_score.values
results["LSS"] = LSS.values

print("Top 15 planets by LSS")
top15 = results.nlargest(15, "LSS")[["pl_name", "LSS", "pl_eqt", "pl_rade", "pl_dens", "pl_orbsmax", "pl_orbeccen"]].round(3)
print(top15.to_string(index=False))

df_ml["LSS"] = LSS.values

df_full = df_ml.copy()
df_full["hz_score"] = hz_score.values
df_full["temp_score"] = temp_score.values
df_full["retention_score"] = retention_score.values
df_full["orbit_score"] = orbit_score.values
df_full["stellar_score"] = stellar_score.values

df_ml.to_csv("exoplanets_step7_model_ready.csv", index=False)
df_full.to_csv("exoplanets_step7_with_lss.csv", index=False)

print(f"Saved -> exoplanets_step7_model_ready.csv ({df_ml.shape[0]} planets, {df_ml.shape[1]} cols)")
print("Saved -> exoplanets_step7_with_lss.csv")
print("Final pipeline")
print("Raw: 6128 x 54")
print("Step 2: 6128 x 15")
print("Step 3: 6128 x 15")
print("Step 4: 5456 x 15")
print("Step 5: 5456 x 15")
print("Step 6: 5456 x 10")
print("Step 7: 5456 x 11")

ML features: (5456, 10)
Raw values: (5456, 15)
Original unit ranges used for LSS
pl_eqt          median=750.00 min=50.00 max=2386.00
pl_rade         median=2.67 min=0.51 max=23.20
pl_dens         median=2.50 min=0.01 max=24.39
pl_orbeccen     median=0.00 min=0.00 max=0.58
pl_orbsmax      median=0.10 min=0.01 max=4.61
st_teff         median=5549.00 min=2960.00 max=8720.00
st_mass         median=0.94 min=0.09 max=2.28
st_age          median=4.27 min=0.00 max=14.00
hz_score mean=0.161 std=0.197 >0.5: 387
temp_score mean=0.257 std=0.221 >0.5: 425
retention mean=0.387 std=0.359 >0.5: 1968
orbit_score mean=0.899 std=0.163 >0.5: 5223
stellar_score mean=0.907 std=0.121 >0.5: 5410
LSS distribution
count    5456.0000
mean        0.3785
std         0.1228
min         0.1390
25%         0.2928
50%         0.3774
75%         0.4386
max         0.9691
dtype: float64
Spread std: 0.1228
Skewness: 0.896
IQR: 0.1457
Earth benchmark
hz    : 0.848
temp  : 1.000
ret   : 0.956
orb   : 0.967
st    : 0.993
Ea

In [ ]:
# STEP 8: Split and scale

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
import joblib

df = pd.read_csv("exoplanets_step7_model_ready.csv")
print(f"Loaded: {df.shape[0]} planets x {df.shape[1]} columns")

FEATURE_COLS = ["pl_rade", "pl_dens", "pl_orbeccen", "pl_eqt", "pl_orbsmax", "st_teff", "st_mass", "st_age"]
TARGET = "LSS"

X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()

print(f"Features: {FEATURE_COLS}")
print(f"Target: {TARGET} (mean={y.mean():.4f}, std={y.std():.4f})")

N_BINS = 5
lss_bins = pd.qcut(y, q=N_BINS, labels=False)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=lss_bins
)

print("TRAIN/TEST SPLIT")
print(f"Total planets: {len(df)}")
print(f"Training set: {len(X_train)} ({len(X_train) / len(df) * 100:.0f}%)")
print(f"Test set: {len(X_test)} ({len(X_test) / len(df) * 100:.0f}%)")

print(f"{'Metric':<12} {'Train':>10} {'Test':>10}")
print("-" * 34)
for metric, fn in [("mean", np.mean), ("std", np.std), ("min", np.min), ("max", np.max), ("median", np.median)]:
    print(f"{metric:<12} {fn(y_train):>10.4f} {fn(y_test):>10.4f}")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=FEATURE_COLS, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=FEATURE_COLS, index=X_test.index)

print("SCALING VERIFICATION (RobustScaler)")
print(f"{'Feature':<15} {'Train Mean':>12} {'Train Std':>10} {'Test Mean':>11} {'Test Std':>10}")
print("-" * 62)
for col in FEATURE_COLS:
    print(f"{col:<15} {X_train_scaled[col].mean():>12.4f} {X_train_scaled[col].std():>10.4f} {X_test_scaled[col].mean():>11.4f} {X_test_scaled[col].std():>10.4f}")

X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)
X_train_scaled.to_csv("X_train_scaled.csv", index=False)
X_test_scaled.to_csv("X_test_scaled.csv", index=False)
joblib.dump(scaler, "robust_scaler.pkl")

print("Saved splits")
print(f"X_train.csv ({X_train.shape[0]} x {X_train.shape[1]})")
print(f"X_test.csv ({X_test.shape[0]} x {X_test.shape[1]})")
print(f"y_train.csv ({len(y_train)} values)")
print(f"y_test.csv ({len(y_test)} values)")
print(f"X_train_scaled.csv ({X_train_scaled.shape[0]} x {X_train_scaled.shape[1]})")
print(f"X_test_scaled.csv ({X_test_scaled.shape[0]} x {X_test_scaled.shape[1]})")
print("robust_scaler.pkl")

print("STEP 8 COMPLETE")
print(f"Split: 80/20 stratified on LSS quintiles")
print(f"Train size: {len(X_train)} planets")
print(f"Test size: {len(X_test)} planets")
print("Scaler: RobustScaler fit on train only")

Loaded: 5456 planets x 11 columns
Features: ['pl_rade', 'pl_dens', 'pl_orbeccen', 'pl_eqt', 'pl_orbsmax', 'st_teff', 'st_mass', 'st_age']
Target: LSS (mean=0.3785, std=0.1228)
TRAIN/TEST SPLIT
Total planets: 5456
Training set: 4364 (80%)
Test set: 1092 (20%)
Metric            Train       Test
----------------------------------
mean             0.3783     0.3795
std              0.1224     0.1243
min              0.1390     0.1400
max              0.9691     0.9533
median           0.3778     0.3758
SCALING VERIFICATION (RobustScaler)
Feature           Train Mean  Train Std   Test Mean   Test Std
--------------------------------------------------------------
pl_rade               0.3901     0.7811      0.4104     0.7766
pl_dens              -0.0386     0.6794     -0.0493     0.7080
pl_orbeccen           0.7924     1.4040      0.7761     1.3698
pl_eqt                0.1629     0.7658      0.1103     0.7536
pl_orbsmax            1.6031     4.8507      1.8514     5.3048
st_teff            